In [1]:
import warnings
warnings.simplefilter(action='ignore')
import os, joblib
import numpy as np
import pandas as pd
import polars as pl
import kaggle_evaluation.default_inference_server
from catboost import CatBoostRegressor, Pool
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.model_selection import train_test_split

In [2]:
train = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')

def preprocessing(data, typ):
    main_features = ['E1','E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 
                     'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9',  
                     'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S12', 
                     "I2", "P8", "P9", "P10", "P12", "P13",
                     
                     "M*", "E*", "I*", "P*", "V*", "S*", "D*",
                     
                     "M*_P*_V*", 'M*_E*_S*', 'M*_P*_I*_V*',
                     'V*_M*_E*_I*', 'V*_S*_D*',
                     'E*_I*_V*_D*', 'M*_V*_S*_D*',
                     'P*_V*_S*', 'P*_M*_D*',
                     'M*_E*_P*_S*', 'M*_E*_I*_P*_V*_S*_D*'
                    ]
    
    data['M*'] = data[[f'M{i}' for i in range(1, 19)]].sum(axis=1).to_numpy()
    data['E*'] = data[[f'E{i}' for i in range(1, 21)]].sum(axis=1).to_numpy()
    data['I*'] = data[[f'I{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()
    data['P*'] = data[['P8', 'P9', 'P10', 'P12', 'P13']].sum(axis=1).to_numpy()
    data['V*'] = data[[f'V{i}' for i in range(1, 14)]].sum(axis=1).to_numpy()
    data['S*'] = data[[f'S{i}' for i in range(1, 13)]].sum(axis=1).to_numpy()
    data['D*'] = data[[f'D{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()

    data[f'ME_prod_M*_E*'] = data['M*'] * data['E*']
    data[f'ME_ratio_M*_E*'] = data['M*'] / (data['E*'] + 1e-8)
            
    main_features.append(f'ME_prod_M*_E*')
    main_features.append(f'ME_ratio_M*_E*')
            
    # M × I (Market × Interest Rate)
    data[f'MI_prod_M*_I*'] = data['M*'] * data['I*']
    data[f'MI_ratio_M*_I*'] = data['M*'] / (data['I*'] + 1e-8)
    data[f'MI_spread_M*_I*'] = data['M*'] - data['I*']
    
    main_features.append(f'MI_prod_M*_I*')
    main_features.append(f'MI_ratio_M*_I*')
    main_features.append(f'MI_spread_M*_I*')
        
    # M × P (Market × Price)
    data[f'MP_prod_M*_P*'] = data['M*'] * data['P*']
    data[f'MP_ratio_M*_P*'] = data['M*'] / (data['P*'] + 1e-8)
    
    main_features.append(f'MP_prod_M*_P*')
    main_features.append(f'MP_ratio_M*_P*')
    
    # M × V (Market × Volatility)
    data[f'MV_ratio_M*_V*'] = data['M*'] / (data['V*'] + 1e-8)
    data[f'MV_prod_M*_V*'] = data['M*'] * data['V*']
    # Sharpe-like ratio
    if 'return' in 'M*'.lower():
        data[f'MV_sharpe_M*_V*'] = data['M*'] / (data['V*'] + 1e-8)
        main_features.append(f'MV_sharpe_M*_V*')
                
    main_features.append(f'MV_ratio_M*_V*')
    main_features.append(f'MV_prod_M*_V*')
        
    # M × S (Market × Sentiment)
    data[f'MS_prod_M*_S*'] = data['M*'] * data['S*']
    data[f'MS_weighted_M*_S*'] = data['M*'] * (1 + data['S*'])
    main_features.append(f'MS_prod_M*_S*')
    main_features.append(f'MS_weighted_M*_S*')
        
    # E × I (Economic × Interest Rate)
    data[f'EI_diff_E*_I*'] = data['E*'] - data['I*']
    data[f'EI_ratio_E*_I*'] = data['E*'] / (data['I*'] + 1e-8)
    data[f'EI_prod_E*_I*'] = data['E*'] * data['I*']
    main_features.append(f'EI_diff_E*_I*')
    main_features.append(f'EI_ratio_E*_I*')
    main_features.append(f'EI_prod_E*_I*')
        
    # E × P (Economic × Price)
    data[f'EP_prod_E*_P*'] = data['E*'] * data['P*']
    data[f'EP_ratio_E*_P*'] = data['E*'] / (data['P*'] + 1e-8)
    main_features.append(f'EP_prod_E*_P*')
    main_features.append(f'EP_ratio_E*_P*')
        
    # E × V (Economic × Volatility)
    data[f'EV_prod_E*_V*'] = data['E*'] * data['V*']
    data[f'EV_uncertainty_E*_V*'] = abs(data['E*']) * data['V*']
    main_features.append(f'EV_prod_E*_V*')
    main_features.append(f'EV_uncertainty_E*_V*')
        
    # E × S (Economic × Sentiment)
    data[f'ES_prod_E*_S*'] = data['E*'] * data['S*']
    data[f'ES_diff_E*_S*'] = data['E*'] - data['S*']
    main_features.append(f'ES_prod_E*_S*')
    main_features.append(f'ES_diff_E*_S*')
        
    # I × P (Interest Rate × Price)
    data[f'IP_prod_I*_P*'] = data['I*'] * data['P*']
    data[f'IP_discount_I*_P*'] = data['P*'] / (1 + data['I*'] + 1e-8)
    main_features.append(f'IP_prod_I*_P*')
    main_features.append(f'IP_discount_I*_P*')
    
    # I × V (Interest Rate × Volatility)
    data[f'IV_prod_I*_V*'] = data['I*'] * data['V*']
    main_features.append(f'IV_prod_I*_V*')
        
    # I × S (Interest Rate × Sentiment)
    data[f'IS_prod_I*_S*'] = data['I*'] * data['S*']
    main_features.append(f'IS_prod_I*_S*')
        
    # P × V (Price × Volatility)
    data[f'PV_prod_P*_V*'] = data['P*'] * data['V*']
    data[f'PV_stability_P*_V*'] = data['P*'] / (data['V*'] + 1e-8)
    main_features.append(f'PV_prod_P*_V*')
    main_features.append(f'PV_stability_P*_V*')
        
    # P × S (Price × Sentiment)
    data[f'PS_prod_P*_S*'] = data['P*'] * data['S*']
    # Contrarian signal (opposite signs amplify)
    data[f'PS_contrarian_P*_S*'] = -data['P*'] * data['S*']
    main_features.append(f'PS_prod_P*_S*')
    main_features.append(f'PS_contrarian_P*_S*')
        
    # V × S (Volatility × Sentiment)
    data[f'VS_prod_V*_S*'] = data['V*'] * data['S*']
    data[f'VS_panic_V*_S*'] = data['V*'] * abs(data['S*'])
    main_features.append(f'VS_prod_V*_S*')
    main_features.append(f'VS_panic_V*_S*')

    # ============================================
    # RISK-ADJUSTED RETURNS (Critical)
    # ============================================
    data['sharpe_proxy'] = data['M*'] / (data['V*'] + 1e-8)
    main_features.append('sharpe_proxy')
    
    data['information_ratio'] = (data['M*'] - data['I*']) / (data['V*'] + 1e-8)
    main_features.append('information_ratio')
    
    # ============================================
    # REGIME INDICATORS (Critical)
    # ============================================
    data['bull_bear_index'] = (data['M*'] + data['S*'] - data['V*']) / 3
    main_features.append('bull_bear_index')
    
    data['regime_classifier'] = np.sign(data['M*']) * np.sign(data['E*'])
    main_features.append('regime_classifier')
    
    # ============================================
    # ECONOMIC STRESS (High Importance)
    # ============================================
    data['economic_stress'] = data['E*'] * data['V*'] * abs(data['S*'])
    main_features.append('economic_stress')
    
    data['recession_signal'] = ((data['E*'] < 0).astype(int)) * data['V*'] * data['I*']
    main_features.append('recession_signal')
    
    # ============================================
    # KEY NON-LINEAR TRANSFORMS
    # ============================================
    data['M*_squared'] = data['M*'] ** 2
    main_features.append('M*_squared')
    
    data['V*_squared'] = data['V*'] ** 2
    main_features.append('V*_squared')
    
    data['log_M*'] = np.sign(data['M*']) * np.log(np.abs(data['M*']) + 1)
    main_features.append('log_M*')
    
    # ============================================
    # THREE-WAY INTERACTIONS (High Signal)
    # ============================================
    data['MEI_prod'] = data['M*'] * data['E*'] * data['I*']
    main_features.append('MEI_prod')
    
    data['MVP_score'] = data['M*'] * data['V*'] * data['P*']
    main_features.append('MVP_score')
    
    # ============================================
    # SENTIMENT-ADJUSTED (Critical for Prediction)
    # ============================================
    data['fear_greed_index'] = (data['S*'] - data['V*']) / (abs(data['S*']) + data['V*'] + 1e-8)
    main_features.append('fear_greed_index')
    
    data['sentiment_strength'] = data['S*'] * data['M*'] / (data['V*'] + 1e-8)
    main_features.append('sentiment_strength')
    
    # ============================================
    # VOLATILITY-ADJUSTED FUNDAMENTALS
    # ============================================
    data['vol_adjusted_E*'] = data['E*'] / (data['V*'] + 1e-8)
    main_features.append('vol_adjusted_E*')
    
    data['vol_adjusted_P*'] = data['P*'] / (data['V*'] + 1e-8)
    main_features.append('vol_adjusted_P*')
    
    # ============================================
    # KEY SPREADS
    # ============================================
    data['ME_spread'] = data['M*'] - data['E*']
    main_features.append('ME_spread')
    
    data['PI_spread'] = data['P*'] - data['I*']
    main_features.append('PI_spread')
    
    # ============================================
    # COMPOSITE QUALITY SCORE
    # ============================================
    data['quality_score'] = (data['E*'] + data['M*']) * data['P*'] / (data['V*'] + abs(data['S*']) + 1e-8)
    main_features.append('quality_score')
    
    # ============================================
    # BINARY SIGNAL AMPLIFIERS
    # ============================================
    data['D*_M*_interaction'] = data['D*'] * data['M*']
    main_features.append('D*_M*_interaction')
    
    data['D*_V*_interaction'] = data['D*'] * data['V*']
    main_features.append('D*_V*_interaction')
    
    data['M*_P*_V*'] = data['M*'] + data['P*'] + data['V*'] 
    data['M*_E*_S*'] = data['M*'] + data['E*'] + data['S*'] 
    data['M*_P*_I*_V*'] = data['M*'] + data['P*'] + data['I*'] + data['V*'] 

    data['V*_M*_E*_I*'] = data['V*'] + data['M*'] + data['E*'] + data['I*'] 
    data['V*_S*_D*'] = data['V*'] + data['S*'] + data['D*'] 

    data['E*_I*_V*_D*'] = data['E*'] + data['I*'] + data['V*'] + data['D*'] 
    data['M*_V*_S*_D*'] = data['M*'] + data['V*'] + data['S*'] + data['D*'] 

    data['P*_V*_S*'] = data['P*'] + data['V*'] + data['S*'] 
    data['P*_M*_D*'] = data['P*'] + data['M*'] + data['D*']

    data['M*_E*_P*_S*'] = data['M*'] + data['E*'] + data['P*'] + data['S*']
    data['M*_E*_I*_P*_V*_S*_D*'] = data['M*'] + data['E*'] + data['I*'] + data['P*'] + data['V*'] + data['S*'] + data['D*']
    
    if typ == "train":
        data = data[main_features + ["forward_returns"]].copy()
        data = data.rename(columns={'forward_returns': 'target'})
    else: 
        data = data[main_features].copy()
    
    if 'target' in data.columns:
        data = data.dropna(subset=['E14'])

    for col in data.columns:
        if col != 'target':
            data[col].fillna(data[col].mean(), inplace=True)
    
    return data

train_processed = preprocessing(train, "train")

train_split, val_split = train_test_split(
    train_processed, test_size=0.1, random_state=42
)

train_x = train_split.drop(columns=["target"])
test_x = val_split.drop(columns=["target"])
train_y = train_split['target']
test_y = val_split['target']

In [3]:
train_processed

,E1,E10,E11,E12,E13,E14,E15,E16,E17,E18,...,fear_greed_index,sentiment_strength,vol_adjusted_E*,vol_adjusted_P*,ME_spread,PI_spread,quality_score,D*_M*_interaction,D*_V*_interaction,target
1006,1.564574,0.597222,0.035053,0.018519,0.005952,0.005952,0.520172,1.116487,0.680294,-0.199948,...,-0.429066,0.708519,2.124151,1.793863,-4.486619,5.684075,10.297146,0.000000,0.000000,-0.000669
1007,1.564574,0.597884,0.034722,0.018188,0.005622,0.005622,0.520503,1.116244,0.681505,-0.198796,...,-0.413073,0.710142,2.028169,1.510344,-4.264789,5.119716,8.199952,0.000000,0.000000,0.005348
1008,1.564574,0.598545,0.034392,0.017857,0.005291,0.005291,0.520833,1.116002,0.682718,-0.197643,...,-0.408727,0.825595,2.194186,1.817310,-4.184551,5.715645,10.392139,0.000000,0.000000,0.001996
1009,1.564574,0.599206,0.034061,0.017526,0.004960,0.004960,0.520503,1.115759,0.683934,-0.196487,...,-0.572721,0.530957,2.129987,1.401218,-4.204592,4.678407,8.939713,0.000000,0.000000,-0.001327
1010,1.564574,0.599868,0.033730,0.017196,0.004630,0.004630,0.520172,1.115517,0.685153,-0.195329,...,-0.586902,0.427668,2.080326,1.487116,-4.599483,5.078860,9.304192,3.285739,6.001323,-0.003987
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8985,1.565379,0.184524,0.019180,0.019180,0.005952,0.005952,0.911376,-0.083496,-0.572447,0.223638,...,-0.094325,0.774132,2.190936,1.492962,-6.674978,0.867861,6.980950,0.000000,0.000000,0.002457
8986,1.562946,0.184193,0.018849,0.018849,0.005622,0.005622,0.911706,-0.083542,-0.572080,0.222910,...,0.217497,3.385677,2.160407,1.714749,-4.850284,1.358154,6.173822,0.000000,0.000000,0.002312
8987,1.560520,0.183862,0.018519,0.018519,0.005291,0.005291,0.912037,-0.083874,-0.572016,0.222211,...,0.216861,2.801633,2.530563,1.459002,-5.191701,-0.137580,5.026194,1.803055,2.764110,0.002891
8988,1.558102,0.183532,0.018188,0.018188,0.004960,0.004960,0.912368,-0.084206,-0.571952,0.221513,...,-0.237235,0.606036,2.478434,2.161599,-6.257880,2.076022,10.997037,0.000000,0.000000,0.008310


In [4]:
import numpy as np
import joblib
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error

class ResidualBoostingEnsemble:
    
    def __init__(self, base_params, n_models=3):
        self.base_params = base_params
        self.n_models = n_models
        self.models = []
        
    def fit(self, train_x, train_y, test_x=None, test_y=None):
        current_train_pred = np.zeros(len(train_y))
        
        for i in range(self.n_models):
            print(f'Training Model {i+1}...')
            
            if i == 0:
                target = train_y
            else:
                residuals = train_y - current_train_pred
                target = residuals
            
            model = LGBMRegressor(**self.base_params)
            model.fit(train_x, target)
            self.models.append(model)
            
            pred = model.predict(train_x)
            current_train_pred += pred
            
            train_mape = mean_absolute_percentage_error(train_y, current_train_pred)
            print(f'Cumulative Training MAPE: {train_mape:.4f}')
            
            if test_x is not None and test_y is not None:
                current_test_pred = sum(m.predict(test_x) for m in self.models)
                test_mape = mean_absolute_percentage_error(test_y, current_test_pred)
                print(f'Cumulative Test MAPE: {test_mape:.4f}')
            
            print()
        
        return self
    
    def predict(self, X):
        predictions = np.zeros(len(X))
        for model in self.models:
            predictions += model.predict(X)
        return predictions
    
    def save(self, filepath):
        joblib.dump(self, filepath)
        print(f'Model saved to {filepath}')
    
    @staticmethod
    def load(filepath):
        return joblib.load(filepath)
    
    def evaluate(self, X, y):
        predictions = self.predict(X)
        mape = mean_absolute_percentage_error(y, predictions)
        mae = mean_absolute_error(y, predictions)
        rmse = np.sqrt(mean_squared_error(y, predictions))
        
        print(f'MAPE: {mape:.4f}')
        print(f'MAE: {mae:.4f}')
        print(f'RMSE: {rmse:.4f}')
        
        return {'mape': mape, 'mae': mae, 'rmse': rmse}


LGBM_R_parm = {
    'boosting_type': 'gbdt', 
    'colsample_bytree': 0.9484106149593443, 
    'learning_rate': 0.1988123373955639, 
    'max_bin': 77, 
    'max_depth': 10, 
    'metric': 'mape', 
    'min_child_samples': 81, 
    'min_data_in_leaf': 21, 
    'n_estimators': 5029, 
    'num_leaves': 42, 
    'objective': 'regression_l1', 
    'reg_alpha': 0.6355835028602363, 
    'reg_lambda': 3.109823217156622, 
    'subsample': 0.7300733288106989, 
    'verbosity': -1
}

ensemble = ResidualBoostingEnsemble(base_params=LGBM_R_parm, n_models=3)
ensemble.fit(train_x, train_y, test_x, test_y)

ensemble.save('LGBM_Residual_Ensemble.pkl')

print('\nFinal Training Evaluation:')
ensemble.evaluate(train_x, train_y)

print('\nFinal Test Evaluation:')
ensemble.evaluate(test_x, test_y)

LGBM_R_model = joblib.load('LGBM_Residual_Ensemble.pkl')

Training Model 1...
Cumulative Training MAPE: 1928719997.4297
Cumulative Test MAPE: 77906156332.2973

Training Model 2...
Cumulative Training MAPE: 2069030730.0660
Cumulative Test MAPE: 74499038387.4211

Training Model 3...
Cumulative Training MAPE: 1690140775.1063
Cumulative Test MAPE: 73155183365.8088

Model saved to LGBM_Residual_Ensemble.pkl

Final Training Evaluation:
MAPE: 1690140775.1063
MAE: 0.0021
RMSE: 0.0050

Final Test Evaluation:
MAPE: 73155183365.8088
MAE: 0.0081
RMSE: 0.0117


In [5]:
SIGNAL_MULTIPLIER = 400.0
MIN_SIGNAL = 0.0
MAX_SIGNAL = 2.0

def convert_ret_to_signal(ret_value):
    return np.clip(ret_value * SIGNAL_MULTIPLIER + 1, MIN_SIGNAL, MAX_SIGNAL)
MIN_INVESTMENT = 0
MAX_INVESTMENT = 2

class ParticipantVisibleError(Exception):
    pass

def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str = None) -> float:
    if not pd.api.types.is_numeric_dtype(submission['prediction']):
        raise ParticipantVisibleError('Predictions must be numeric')

    solution = solution.copy()
    solution['position'] = submission['prediction'].values

    if solution['position'].max() > MAX_INVESTMENT:
        raise ParticipantVisibleError(f'Position of {solution["position"].max()} exceeds maximum of {MAX_INVESTMENT}')
    if solution['position'].min() < MIN_INVESTMENT:
        raise ParticipantVisibleError(f'Position of {solution["position"].min()} below minimum of {MIN_INVESTMENT}')

    solution['strategy_returns'] = solution['risk_free_rate'] * (1 - solution['position']) + solution['position'] * solution['forward_returns']

    strategy_excess_returns = solution['strategy_returns'] - solution['risk_free_rate']
    strategy_excess_cumulative = (1 + strategy_excess_returns).prod()
    strategy_mean_excess_return = (strategy_excess_cumulative) ** (1 / len(solution)) - 1
    strategy_std = solution['strategy_returns'].std()

    trading_days_per_yr = 252
    if strategy_std == 0:
        raise ParticipantVisibleError('Division by zero, strategy std is zero')
    sharpe = strategy_mean_excess_return / strategy_std * np.sqrt(trading_days_per_yr)
    strategy_volatility = float(strategy_std * np.sqrt(trading_days_per_yr) * 100)

    market_excess_returns = solution['forward_returns'] - solution['risk_free_rate']
    market_excess_cumulative = (1 + market_excess_returns).prod()
    market_mean_excess_return = (market_excess_cumulative) ** (1 / len(solution)) - 1
    market_std = solution['forward_returns'].std()

    market_volatility = float(market_std * np.sqrt(trading_days_per_yr) * 100)

    if market_volatility == 0:
        raise ParticipantVisibleError('Division by zero, market std is zero')

    excess_vol = max(0, strategy_volatility / market_volatility - 1.2) if market_volatility > 0 else 0
    vol_penalty = 1 + excess_vol

    return_gap = max(
        0,
        (market_mean_excess_return - strategy_mean_excess_return) * 100 * trading_days_per_yr,
    )
    return_penalty = 1 + (return_gap**2) / 100

    adjusted_sharpe = sharpe / (vol_penalty * return_penalty)
    return min(float(adjusted_sharpe), 1_000_000)


print("\nEvaluating on validation set...")
val_predictions = LGBM_R_model.predict(test_x)
val_signals = np.array([convert_ret_to_signal(pred) for pred in val_predictions])

val_indices = val_split.index
val_original = train.loc[val_indices].copy()

solution_df = pd.DataFrame({
    'forward_returns': val_original['forward_returns'].values,
    'risk_free_rate': val_original['risk_free_rate'].values
})

submission_df = pd.DataFrame({
    'prediction': val_signals
})

try:
    validation_score = score(solution_df, submission_df)
    print(f"Validation Adjusted Sharpe Ratio: {validation_score}")
except Exception as e:
    print(f"Error calculating validation score: {e}")


Evaluating on validation set...
Validation Adjusted Sharpe Ratio: 0.3299182256349373


In [6]:
def predict(test: pl.DataFrame) -> float:
    try:
        test_df = test.to_pandas()
        
        test_processed = preprocessing(test_df, 'inference')
        
        LGBM_R_y_pred = LGBM_R_model.predict(test_processed)
        signal = convert_ret_to_signal(LGBM_R_y_pred)
        
        return float(signal)
        
    except Exception as e:
        print(f"Prediction error: {e}")
        return 1.0 
    
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))